<a href="https://www.kaggle.com/code/muhammaddhiyaulatha/arc-baseline-zero-model-ipynb?scriptVersionId=313475789" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# ARC Prize 2026 - Baseline Model

This notebook contains my first submission to the ARC-AGI-2 competition on Kaggle.

## Approach

* Generate output grids filled with zeros
* Match input grid dimensions
* Ensure correct submission format

## Purpose

This is a baseline to understand:

* Submission pipeline
* Evaluation system
* Dataset structure

Next step: implement rule-based reasoning.


In [1]:
import json
import os
from collections import Counter, defaultdict, deque
from itertools import product, chain
from typing import List, Tuple, Dict, Any, Optional, Callable, Set
import numpy as np
from functools import lru_cache
import math

# ============================================================================
# 0. GLOBAL CONFIGURATION & CACHING
# ============================================================================

IS_RERUN = bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN"))
PATH = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json" if IS_RERUN else "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

with open(PATH) as f:
    DATA = json.load(f)

# Global caches for performance
GRID_CACHE: Dict[Tuple, Any] = {}
TRANSFORM_CACHE: Dict[Tuple, Any] = {}
PATTERN_CACHE: Dict[Tuple, Any] = {}

# ============================================================================
# 1. ULTRA-FAST GRID UTILITIES (Optimized with NumPy)
# ============================================================================

def grid_to_np(grid: List[List[int]]) -> np.ndarray:
    """Convert grid to numpy array for fast operations."""
    return np.array(grid, dtype=np.uint8)

def np_to_grid(arr: np.ndarray) -> List[List[int]]:
    """Convert numpy array back to grid."""
    return arr.tolist()

def copy_grid(g: List[List[int]]) -> List[List[int]]:
    """Fast grid copy using list comprehension."""
    return [row[:] for row in g]

def grid_dims(g: List[List[int]]) -> Tuple[int, int]:
    """Get grid dimensions with error checking."""
    if not g:
        return 0, 0
    return len(g), len(g[0])

def flatten_grid(g: List[List[int]]) -> List[int]:
    """Flatten grid using itertools chain."""
    return list(chain.from_iterable(g))

def grids_equal(a: List[List[int]], b: List[List[int]]) -> bool:
    """Fast grid equality check using string comparison."""
    if len(a) != len(b) or len(a[0]) != len(b[0]):
        return False
    return all(ra == rb for ra, rb in zip(a, b))

def unique_colors(g: List[List[int]]) -> Set[int]:
    """Get unique colors using set comprehension."""
    return {c for row in g for c in row}

def most_common_color(grid: List[List[int]]) -> int:
    """Fast most common color using Counter."""
    return Counter(flatten_grid(grid)).most_common(1)[0][0]

def background_color(grid: List[List[int]]) -> int:
    """Get background color (most common)."""
    return most_common_color(grid)

def color_counts(grid: List[List[int]]) -> Counter:
    """Count colors efficiently."""
    return Counter(flatten_grid(grid))

def grid_hash(g: List[List[int]]) -> int:
    """Fast grid hashing for caching."""
    return hash(tuple(tuple(row) for row in g))

# ============================================================================
# 2. PRECOMPUTED GEOMETRIC TRANSFORMS (Cached)
# ============================================================================

@lru_cache(maxsize=None)
def rotate90_cached(grid_tuple: Tuple[Tuple[int, ...], ...]) -> Tuple[Tuple[int, ...], ...]:
    """Cached rotate 90 degrees."""
    return tuple(zip(*grid_tuple[::-1]))

def rotate90(g: List[List[int]]) -> List[List[int]]:
    """Rotate grid 90 degrees clockwise."""
    return np_to_grid(np.rot90(grid_to_np(g), -1))

def rotate180(g: List[List[int]]) -> List[List[int]]:
    """Rotate grid 180 degrees."""
    return np_to_grid(np.rot90(grid_to_np(g), 2))

def rotate270(g: List[List[int]]) -> List[List[int]]:
    """Rotate grid 270 degrees clockwise."""
    return np_to_grid(np.rot90(grid_to_np(g), 1))

def flip_h(g: List[List[int]]) -> List[List[int]]:
    """Flip horizontally."""
    return np_to_grid(np.fliplr(grid_to_np(g)))

def flip_v(g: List[List[int]]) -> List[List[int]]:
    """Flip vertically."""
    return np_to_grid(np.flipud(grid_to_np(g)))

def transpose(g: List[List[int]]) -> List[List[int]]:
    """Transpose grid."""
    return np_to_grid(np.transpose(grid_to_np(g)))

def transpose_anti(g: List[List[int]]) -> List[List[int]]:
    """Anti-transpose (rotate 90 then flip vertical)."""
    return flip_v(rotate90(g))

ALL_RIGID_TRANSFORMS = [
    ("identity", lambda x: copy_grid(x)),
    ("rot90", rotate90),
    ("rot180", rotate180),
    ("rot270", rotate270),
    ("flip_h", flip_h),
    ("flip_v", flip_v),
    ("transpose", transpose),
    ("transpose_anti", transpose_anti),
]

# ============================================================================
# 3. OPTIMIZED COLOR MAPPING
# ============================================================================

def color_map(g: List[List[int]], mapping: Dict[int, int]) -> List[List[int]]:
    """Fast color mapping using numpy."""
    arr = grid_to_np(g)
    # Create lookup table
    max_color = max(arr.max(), max(mapping.keys(), default=0))
    lut = np.array([mapping.get(i, i) for i in range(max_color + 1)], dtype=np.uint8)
    return np_to_grid(lut[arr])

def infer_color_map(train: List[Dict]) -> Dict[int, int]:
    """Infer color mapping from training pairs."""
    mapping = {}
    for pair in train:
        inp_arr = grid_to_np(pair["input"])
        out_arr = grid_to_np(pair["output"])
        h = min(inp_arr.shape[0], out_arr.shape[0])
        w = min(inp_arr.shape[1], out_arr.shape[1])
        
        for i in range(h):
            for j in range(w):
                inp_color = inp_arr[i, j]
                out_color = out_arr[i, j]
                if inp_color in mapping and mapping[inp_color] != out_color:
                    # Conflict found, try to resolve
                    continue
                mapping[inp_color] = out_color
    return mapping

# ============================================================================
# 4. VECTORIZED CROPPING & BOUNDING BOX
# ============================================================================

def bounding_box(grid: List[List[int]], ignore_color: Optional[int] = None) -> Tuple[int, int, int, int]:
    """Find bounding box of non-background cells using numpy."""
    arr = grid_to_np(grid)
    if ignore_color is None:
        ignore_color = background_color(grid)
    
    mask = arr != ignore_color
    if not np.any(mask):
        return 0, 0, arr.shape[0], arr.shape[1]
    
    rows = np.where(mask.any(axis=1))[0]
    cols = np.where(mask.any(axis=0))[0]
    
    if rows.size == 0 or cols.size == 0:
        return 0, 0, arr.shape[0], arr.shape[1]
    
    return rows[0], cols[0], rows[-1] + 1, cols[-1] + 1

def crop(grid: List[List[int]], r1: int, c1: int, r2: int, c2: int) -> List[List[int]]:
    """Crop grid using numpy slicing."""
    arr = grid_to_np(grid)
    return np_to_grid(arr[r1:r2, c1:c2])

def crop_to_content(grid: List[List[int]], ignore_color: Optional[int] = None) -> List[List[int]]:
    """Crop to content using bounding box."""
    r1, c1, r2, c2 = bounding_box(grid, ignore_color)
    return crop(grid, r1, c1, r2, c2)

def crop_color(grid: List[List[int]], color: int) -> List[List[int]]:
    """Crop to bounding box of specific color."""
    arr = grid_to_np(grid)
    mask = arr == color
    if not np.any(mask):
        return [[]]
    
    rows = np.where(mask.any(axis=1))[0]
    cols = np.where(mask.any(axis=0))[0]
    
    return np_to_grid(arr[rows[0]:rows[-1]+1, cols[0]:cols[-1]+1])

# ============================================================================
# 5. NUMPY-OPTIMIZED TILING & SCALING
# ============================================================================

def tile_grid(g: List[List[int]], reps_h: int, reps_w: int) -> List[List[int]]:
    """Tile grid using numpy.tile."""
    return np_to_grid(np.tile(grid_to_np(g), (reps_h, reps_w)))

def scale_grid(g: List[List[int]], factor_h: int, factor_w: int) -> List[List[int]]:
    """Scale grid using numpy.repeat."""
    arr = grid_to_np(g)
    arr = np.repeat(arr, factor_w, axis=1)
    arr = np.repeat(arr, factor_h, axis=0)
    return np_to_grid(arr)

def downscale_grid(g: List[List[int]], factor_h: int, factor_w: int) -> List[List[int]]:
    """Downscale grid using block reduction."""
    arr = grid_to_np(g)
    h, w = arr.shape
    nh, nw = h // factor_h, w // factor_w
    
    # Reshape and find mode of each block
    result = []
    for i in range(nh):
        row = []
        for j in range(nw):
            block = arr[i*factor_h:(i+1)*factor_h, j*factor_w:(j+1)*factor_w]
            row.append(np.argmax(np.bincount(block.flatten())))
        result.append(row)
    return result

# ============================================================================
# 6. OPTIMIZED CONNECTED COMPONENTS (BFS with deque)
# ============================================================================

def flood_fill_positions(grid: List[List[int]], start_r: int, start_c: int, 
                         color: Optional[int] = None, visited: Optional[Set[Tuple[int, int]]] = None) -> List[Tuple[int, int]]:
    """Flood fill using BFS with deque."""
    h, w = grid_dims(grid)
    if color is None:
        color = grid[start_r][start_c]
    if visited is None:
        visited = set()
    
    queue = deque([(start_r, start_c)])
    positions = []
    
    while queue:
        r, c = queue.popleft()
        if (r, c) in visited or r < 0 or r >= h or c < 0 or c >= w:
            continue
        if grid[r][c] != color:
            continue
        
        visited.add((r, c))
        positions.append((r, c))
        
        # 4-directional neighbors
        for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            queue.append((r + dr, c + dc))
    
    return positions

def connected_components(grid: List[List[int]], ignore_color: Optional[int] = None) -> List[Tuple[int, List[Tuple[int, int]]]]:
    """Find all connected components."""
    h, w = grid_dims(grid)
    visited = set()
    components = []
    
    for i in range(h):
        for j in range(w):
            if (i, j) not in visited:
                if ignore_color is not None and grid[i][j] == ignore_color:
                    visited.add((i, j))
                    continue
                
                color = grid[i][j]
                positions = flood_fill_positions(grid, i, j, color, visited)
                components.append((color, positions))
    
    return components

def extract_objects(grid: List[List[int]], bg: Optional[int] = None) -> List[Dict]:
    """Extract objects as connected components."""
    if bg is None:
        bg = background_color(grid)
    
    comps = connected_components(grid, ignore_color=bg)
    objects = []
    
    for color, positions in comps:
        if not positions:
            continue
            
        min_r = min(r for r, _ in positions)
        max_r = max(r for r, _ in positions)
        min_c = min(c for _, c in positions)
        max_c = max(c for _, c in positions)
        
        oh = max_r - min_r + 1
        ow = max_c - min_c + 1
        
        obj_grid = [[bg] * ow for _ in range(oh)]
        for r, c in positions:
            obj_grid[r - min_r][c - min_c] = color
        
        objects.append({
            "color": color,
            "positions": positions,
            "grid": obj_grid,
            "bbox": (min_r, min_c, max_r + 1, max_c + 1),
            "size": len(positions),
            "height": oh,
            "width": ow
        })
    
    return objects

# ============================================================================
# 7. ADVANCED PATTERN RECOGNITION
# ============================================================================

def detect_pattern(grid: List[List[int]]) -> Dict[str, Any]:
    """Detect grid patterns and symmetries."""
    h, w = grid_dims(grid)
    arr = grid_to_np(grid)
    
    patterns = {
        "is_horizontal_symmetric": np.array_equal(arr, flip_h(arr)),
        "is_vertical_symmetric": np.array_equal(arr, flip_v(arr)),
        "is_diagonal_symmetric": np.array_equal(arr, transpose(arr)),
        "is_rotational_symmetric": np.array_equal(arr, rotate180(arr)),
        "has_regular_pattern": False,
        "repeating_unit": None,
        "grid_dividers": None
    }
    
    # Check for repeating patterns
    for uh in range(1, h // 2 + 1):
        if h % uh != 0:
            continue
        for uw in range(1, w // 2 + 1):
            if w % uw != 0:
                continue
            
            unit = np_to_grid(arr[:uh, :uw])
            tiled = tile_grid(unit, h // uh, w // uw)
            if grids_equal(tiled, grid):
                patterns["has_regular_pattern"] = True
                patterns["repeating_unit"] = unit
                patterns["unit_size"] = (uh, uw)
                break
    
    # Detect grid dividers
    h_dividers = []
    for i in range(h):
        if len(set(grid[i])) == 1:
            h_dividers.append(i)
    
    v_dividers = []
    for j in range(w):
        if len(set(arr[:, j])) == 1:
            v_dividers.append(j)
    
    patterns["grid_dividers"] = (h_dividers, v_dividers)
    patterns["is_single_color"] = len(unique_colors(grid)) == 1
    
    return patterns

# ============================================================================
# 8. COMPOSITE OPERATIONS
# ============================================================================

def apply_composite_operation(grid: List[List[int]], 
                             operations: List[Tuple[str, Any]]) -> List[List[int]]:
    """Apply multiple operations in sequence."""
    result = copy_grid(grid)
    for op_name, op_params in operations:
        if op_name == "rotate":
            result = globals()[f"rotate{op_params}"](result)
        elif op_name == "flip":
            result = globals()[f"flip_{op_params}"](result)
        elif op_name == "scale":
            result = scale_grid(result, op_params[0], op_params[1])
        elif op_name == "color_map":
            result = color_map(result, op_params)
        elif op_name == "crop":
            result = crop_to_content(result, op_params)
        elif op_name == "gravity":
            result = globals()[f"gravity_{op_params}"](result)
    return result

# ============================================================================
# 9. INTELLIGENT STRATEGY MATCHING
# ============================================================================

class Strategy:
    """Strategy base class for pattern matching."""
    
    def __init__(self, name: str, priority: int = 1):
        self.name = name
        self.priority = priority
        self.confidence = 0.0
    
    def match(self, train: List[Dict]) -> bool:
        """Check if strategy matches training examples."""
        raise NotImplementedError
    
    def apply(self, grid: List[List[int]]) -> List[List[int]]:
        """Apply strategy to grid."""
        raise NotImplementedError

class RigidTransformStrategy(Strategy):
    """Match rigid transforms."""
    
    def __init__(self):
        super().__init__("rigid_transform", priority=2)
    
    def match(self, train: List[Dict]) -> bool:
        for _, op in ALL_RIGID_TRANSFORMS:
            if all(grids_equal(op(p["input"]), p["output"]) for p in train):
                self.transform = op
                self.confidence = 1.0
                return True
        return False
    
    def apply(self, grid: List[List[int]]) -> List[List[int]]:
        return self.transform(grid)

class ColorMappingStrategy(Strategy):
    """Match color mappings."""
    
    def __init__(self):
        super().__init__("color_mapping", priority=3)
    
    def match(self, train: List[Dict]) -> bool:
        mapping = infer_color_map(train)
        if not mapping:
            return False
        
        identity = all(mapping.get(k, k) == k for k in mapping)
        if identity:
            return False
        
        if all(grids_equal(color_map(p["input"], mapping), p["output"]) for p in train):
            self.mapping = mapping
            self.confidence = 1.0
            return True
        return False
    
    def apply(self, grid: List[List[int]]) -> List[List[int]]:
        return color_map(grid, self.mapping)

class ScaleStrategy(Strategy):
    """Match scaling operations."""
    
    def __init__(self):
        super().__init__("scaling", priority=4)
    
    def match(self, train: List[Dict]) -> bool:
        for fh in range(1, 8):
            for fw in range(1, 8):
                if fh == 1 and fw == 1:
                    continue
                
                if all(
                    len(p["output"]) == len(p["input"]) * fh and
                    len(p["output"][0]) == len(p["input"][0]) * fw and
                    grids_equal(scale_grid(p["input"], fh, fw), p["output"])
                    for p in train
                ):
                    self.factors = (fh, fw)
                    self.confidence = 1.0
                    return True
        return False
    
    def apply(self, grid: List[List[int]]) -> List[List[int]]:
        return scale_grid(grid, self.factors[0], self.factors[1])

# Add more strategy classes...

# ============================================================================
# 10. STRATEGY MANAGER
# ============================================================================

class StrategyManager:
    """Manage and prioritize strategies."""
    
    def __init__(self):
        self.strategies = [
            RigidTransformStrategy(),
            ColorMappingStrategy(),
            ScaleStrategy(),
            # Add more strategies here...
        ]
        self.strategies.sort(key=lambda s: s.priority, reverse=True)
    
    def find_best_strategy(self, train: List[Dict]) -> Optional[Strategy]:
        """Find the best matching strategy for training data."""
        for strategy in self.strategies:
            try:
                if strategy.match(train):
                    return strategy
            except Exception:
                continue
        return None

# ============================================================================
# 11. ENSEMBLE PREDICTION
# ============================================================================

class EnsemblePredictor:
    """Make predictions using ensemble of strategies."""
    
    def __init__(self):
        self.strategy_manager = StrategyManager()
        self.fallback_strategies = [
            lambda g: copy_grid(g),
            lambda g: [[background_color(g)] * len(g[0]) for _ in range(len(g))],
            lambda g: crop_to_content(g),
            lambda g: make_4way_symmetric(g),
        ]
    
    def predict(self, train: List[Dict], test_input: List[List[int]]) -> List[List[List[int]]]:
        """Generate multiple predictions for a test input."""
        predictions = []
        
        # Try primary strategy
        strategy = self.strategy_manager.find_best_strategy(train)
        if strategy:
            predictions.append(strategy.apply(test_input))
        
        # Add fallback predictions
        for fallback in self.fallback_strategies:
            try:
                predictions.append(fallback(test_input))
            except Exception:
                continue
        
        # Ensure unique predictions
        unique_predictions = []
        seen = set()
        for pred in predictions:
            pred_hash = grid_hash(pred)
            if pred_hash not in seen:
                seen.add(pred_hash)
                unique_predictions.append(pred)
        
        # Pad to at least 2 predictions
        while len(unique_predictions) < 2:
            unique_predictions.append(copy_grid(test_input))
        
        return unique_predictions[:2]  # Return only 2 attempts

# ============================================================================
# 12. MAIN SOLVER
# ============================================================================

def analyze_task(task: Dict) -> Dict[str, Any]:
    """Analyze task for patterns and features."""
    train = task["train"]
    test = task["test"]
    
    analysis = {
        "train_count": len(train),
        "test_count": len(test),
        "input_dims": [grid_dims(p["input"]) for p in train],
        "output_dims": [grid_dims(p["output"]) for p in train],
        "color_counts": [len(unique_colors(p["input"])) for p in train],
        "patterns": [],
        "difficulty": "unknown"
    }
    
    # Analyze patterns
    for i, p in enumerate(train):
        pattern = detect_pattern(p["input"])
        analysis["patterns"].append(pattern)
    
    # Determine difficulty
    if all(grid_dims(p["input"]) == grid_dims(p["output"]) for p in train):
        analysis["difficulty"] = "simple_transform"
    elif all(len(p["output"]) > len(p["input"]) for p in train):
        analysis["difficulty"] = "expansion"
    elif all(len(p["output"]) < len(p["input"]) for p in train):
        analysis["difficulty"] = "reduction"
    else:
        analysis["difficulty"] = "complex"
    
    return analysis

def solve_task_optimized(task: Dict) -> List[Dict]:
    """Solve task with optimized approach."""
    predictor = EnsemblePredictor()
    results = []
    
    for test_case in task["test"]:
        inp = test_case["input"]
        predictions = predictor.predict(task["train"], inp)
        
        results.append({
            "attempt_1": predictions[0] if len(predictions) > 0 else copy_grid(inp),
            "attempt_2": predictions[1] if len(predictions) > 1 else [[background_color(inp)] * len(inp[0]) for _ in range(len(inp))]
        })
    
    return results

# ============================================================================
# 13. MAIN EXECUTION
# ============================================================================

def main():
    """Main execution function."""
    submission = {}
    total_tasks = len(DATA)
    
    print(f"🔍 Processing {total_tasks} tasks...")
    
    for idx, (task_id, task) in enumerate(DATA.items()):
        if idx % 10 == 0:
            print(f"  Progress: {idx}/{total_tasks}")
        
        submission[task_id] = solve_task_optimized(task)
    
    with open("submission.json", "w") as f:
        json.dump(submission, f)
    
    print("✅ submission.json created successfully!")
    print(f"📊 Processed {total_tasks} tasks")

if __name__ == "__main__":
    main()

🔍 Processing 120 tasks...
  Progress: 0/120
  Progress: 10/120
  Progress: 20/120
  Progress: 30/120
  Progress: 40/120
  Progress: 50/120
  Progress: 60/120
  Progress: 70/120
  Progress: 80/120
  Progress: 90/120
  Progress: 100/120
  Progress: 110/120
✅ submission.json created successfully!
📊 Processed 120 tasks
